In [1]:
import sys
import os

current_dir = os.getcwd()
if 'data_preprocessing' in current_dir or 'tests' in current_dir:
    src_dir = os.path.dirname(current_dir)
else:
    src_dir = os.path.join(current_dir, 'samples', 'order_flow_ray', 'src')
sys.path.append(src_dir)

import polars as pl
from pipeline.config import PipelineConfig, DataConfig, ProcessingConfig, StorageConfig, RayConfig, S3Location
from pipeline.pipeline import Pipeline
from data_preprocessing.data_normalization import BMLLNormalizer
from data_preprocessing.data_access import DataAccessFactory

In [19]:
# 1. Create pipeline

config = PipelineConfig(
    region='us-east-1',
    data=DataConfig(
        raw_data_path='s3://bmlldata',
        start_date='2024-01-02',
        end_date='2024-01-02',
        exchanges=['ARCX'],
        data_types=['trades']
    ),
    processing=ProcessingConfig(
        normalization=BMLLNormalizer()
    ),
    storage=StorageConfig(
        raw_data=S3Location(path='s3://bmlldata'),
        normalized=S3Location(path='s3://orderflowanalysis/intermediate/normalized'),
        features=S3Location(path='s3://orderflowanalysis/intermediate/features'),
        models=S3Location(path='s3://orderflowanalysis/output/models'),
        predictions=S3Location(path='s3://orderflowanalysis/output/predictions'),
        backtest=S3Location(path='s3://orderflowanalysis/output/backtest')
    ),
    ray=RayConfig(runtime_env={"working_dir": src_dir}, cpu_buffer=5)
)

pipeline = Pipeline(config)

In [ ]:
# 2. Run pipeline
# Option 1: Run with max_files limit
# results = pipeline.run(max_files=5)

# Option 2: Run with specific file (same as test_s3tables.py)
results= None
results = pipeline.run(max_files=18_000)
print(f"Processed {len(results)} files")

2026-01-19 03:29:31,385	INFO worker.py:1821 -- Connecting to existing Ray cluster at address: 10.0.22.251:6379...
2026-01-19 03:29:31,391	INFO worker.py:1998 -- Connected to Ray cluster. View the dashboard at https://session-tv6mdq12al4ja374phi8ycm5f2.i.anyscaleuserdata.com 
2026-01-19 03:29:31,402	INFO packaging.py:691 -- Creating a file package for local module '/home/ray/default/quant-research-sample-using-amazon-ecs-and-aws-batch/samples/order_flow_ray/src'.
2026-01-19 03:29:31,416	INFO packaging.py:463 -- Pushing file package 'gcs://_ray_pkg_608dd97ebabcdccc.zip' (1.27MiB) to Ray cluster...
2026-01-19 03:29:31,419	INFO packaging.py:476 -- Successfully pushed file package 'gcs://_ray_pkg_608dd97ebabcdccc.zip'.


Discovered 15694 files
Running normalization...
(raylet) Task normalize_file failed. There are infinite retries remaining, so the task will be retried. Error: Task was killed due to the node running low on memory.
Memory on the node (IP: 10.0.46.202, ID: 8479104b62590a193b58fb35cfc82d59ccfa31ee8b891a6ff6252b3c) where the lease (actor ID: NIL_IDlease ID: 5ff2690009000000ffffffffffffffffffffffffffffffffffffffffffffffff, name=normalize_file, pid=21948, memory used=0.54GB) was running was 349.85GB / 368.00GB (0.950686), which exceeds the memory usage threshold of 0.95. Ray killed this worker (ID: 0b263d0faec682d62d5fc465b0b51bb10dd1e21b340a9333c0ccfb3b) because it was the most recently scheduled task; to see more information about memory usage on this node, use `ray logs raylet.out -ip 10.0.46.202`. To see the logs of the worker, use `ray logs worker-0b263d0faec682d62d5fc465b0b51bb10dd1e21b340a9333c0ccfb3b*out -ip 10.0.46.202. Top 10 memory users:
PID	MEM(GB)	COMMAND
24041	25.83	ray::norma

(raylet, ip=10.0.34.15) {"asctime":"2026-01-19 03:30:28,807","levelname":"E","message":"1 Workers (tasks / actors) killed due to memory pressure (OOM), 0 Workers crashed due to other reasons at node (ID: 9fc0bcb715f7a14564684a65142f4b84252f7cf2492903a7dee10fe6, IP: 10.0.34.15) over the last time period. To see more information about the Workers killed on this node, use `ray logs raylet.out -ip 10.0.34.15`\n\nRefer to the documentation on how to address the out of memory issue: https://docs.ray.io/en/latest/ray-core/scheduling/ray-oom-prevention.html. Consider provisioning more memory on this node or reducing task parallelism by requesting more CPUs per task. To adjust the kill threshold, set the environment variable `RAY_memory_usage_threshold` when starting Ray. To disable worker killing, set the environment variable `RAY_memory_monitor_refresh_ms` to zero.","component":"raylet","filename":"node_manager.cc","lineno":3256}


(raylet) A worker died or was killed while executing a task by an unexpected system error. To troubleshoot the problem, check the logs for the dead worker. Lease ID: a0bf690009000000ffffffffffffffffffffffffffffffffffffffffffffffff Worker ID: e24e3eb486d2c29bd10e84bcebb97ed8aa229b8251cbb3e7c017e4b8 Node ID: 117fd100e664b867c549509f612f463ed8dc9e74d7983370478d7953 Worker IP address: 10.0.40.206 Worker port: 10059 Worker PID: 20014 Worker exit type: SYSTEM_ERROR Worker exit detail: The leased worker has unrecoverable failure. Worker is requested to be destroyed when it is returned. RPC error: Socket closed
(raylet) Task normalize_file failed. There are 2 retries remaining, so the task will be retried. Error: 
(raylet) Task normalize_file failed. There are infinite retries remaining, so the task will be retried. Error: Task was killed due to the node running low on memory.
Memory on the node (IP: 10.0.35.88, ID: c241ebc202da5465fb858be646a18a94a843c9375d0a8550164ade56) where the lease (act

(raylet, ip=10.0.41.44) {"asctime":"2026-01-19 03:30:37,161","levelname":"E","message":"1 Workers (tasks / actors) killed due to memory pressure (OOM), 0 Workers crashed due to other reasons at node (ID: 2e257248de6c1421b589823d09e767b989d800b99ba02028d8b80107, IP: 10.0.41.44) over the last time period. To see more information about the Workers killed on this node, use `ray logs raylet.out -ip 10.0.41.44`\n\nRefer to the documentation on how to address the out of memory issue: https://docs.ray.io/en/latest/ray-core/scheduling/ray-oom-prevention.html. Consider provisioning more memory on this node or reducing task parallelism by requesting more CPUs per task. To adjust the kill threshold, set the environment variable `RAY_memory_usage_threshold` when starting Ray. To disable worker killing, set the environment variable `RAY_memory_monitor_refresh_ms` to zero.","component":"raylet","filename":"node_manager.cc","lineno":3256} [repeated 2x across cluster]


...

(raylet) Task normalize_file failed. There are infinite retries remaining, so the task will be retried. Error: Task was killed due to the node running low on memory.
Memory on the node (IP: 10.0.44.88, ID: 7fb46d915d4d780c9d49d4eeeda6b4df4ce0342c12f60a8eadb3fbc0) where the lease (actor ID: NIL_IDlease ID: e6ba820009000000ffffffffffffffffffffffffffffffffffffffffffffffff, name=normalize_file, pid=30480, memory used=6.36GB) was running was 349.88GB / 368.00GB (0.950773), which exceeds the memory usage threshold of 0.95. Ray killed this worker (ID: 6a8ca771bfdad6e5a44c4fe43a4e0f4e881ebff71dc59553afa8e9ad) because it was the most recently scheduled task; to see more information about memory usage on this node, use `ray logs raylet.out -ip 10.0.44.88`. To see the logs of the worker, use `ray logs worker-6a8ca771bfdad6e5a44c4fe43a4e0f4e881ebff71dc59553afa8e9ad*out -ip 10.0.44.88. Top 10 memory users:
PID	MEM(GB)	COMMAND
27512	54.59	ray::normalize_file
27228	53.59	ray::normalize_file
27511	50.

(raylet, ip=10.0.45.52) {"asctime":"2026-01-19 03:33:23,829","levelname":"E","message":"1 Workers (tasks / actors) killed due to memory pressure (OOM), 0 Workers crashed due to other reasons at node (ID: 16e15d96ad2704972651f4cfe94ab11b3caf2442717cca30ef54bf17, IP: 10.0.45.52) over the last time period. To see more information about the Workers killed on this node, use `ray logs raylet.out -ip 10.0.45.52`\n\nRefer to the documentation on how to address the out of memory issue: https://docs.ray.io/en/latest/ray-core/scheduling/ray-oom-prevention.html. Consider provisioning more memory on this node or reducing task parallelism by requesting more CPUs per task. To adjust the kill threshold, set the environment variable `RAY_memory_usage_threshold` when starting Ray. To disable worker killing, set the environment variable `RAY_memory_monitor_refresh_ms` to zero.","component":"raylet","filename":"node_manager.cc","lineno":3256} [repeated 14x across cluster]


(raylet) Task normalize_file failed. There are infinite retries remaining, so the task will be retried. Error: Task was killed due to the node running low on memory.
Memory on the node (IP: 10.0.35.88, ID: c241ebc202da5465fb858be646a18a94a843c9375d0a8550164ade56) where the lease (actor ID: NIL_IDlease ID: 2e0d840009000000ffffffffffffffffffffffffffffffffffffffffffffffff, name=normalize_file, pid=34859, memory used=14.06GB) was running was 349.80GB / 368.00GB (0.950536), which exceeds the memory usage threshold of 0.95. Ray killed this worker (ID: 926836dd65a2e727699d79d6d721fc494d200e5bebb6722d1d092cf2) because it was the most recently scheduled task; to see more information about memory usage on this node, use `ray logs raylet.out -ip 10.0.35.88`. To see the logs of the worker, use `ray logs worker-926836dd65a2e727699d79d6d721fc494d200e5bebb6722d1d092cf2*out -ip 10.0.35.88. Top 10 memory users:
PID	MEM(GB)	COMMAND
30884	48.10	ray::normalize_file
30885	43.48	ray::normalize_file
31172	37

(raylet, ip=10.0.54.194) {"asctime":"2026-01-19 03:33:24,548","levelname":"E","message":"Some workers of the worker process(16572) have not registered within the timeout. The process is still alive, probably it's hanging during start.","component":"raylet","filename":"worker_pool.cc","lineno":586}
(raylet, ip=10.0.54.194) {"asctime":"2026-01-19 03:33:24,582","levelname":"E","message":"Broken Pipe happened during calling ServerConnection::DoAsyncWrites.","component":"raylet","filename":"client_connection.cc","lineno":295}


(raylet) Task normalize_file failed. There are infinite retries remaining, so the task will be retried. Error: Task was killed due to the node running low on memory.
Memory on the node (IP: 10.0.32.18, ID: 9c24aee147b2c080676da313d8af3d0e459066db89463400d1765456) where the lease (actor ID: NIL_IDlease ID: f70c830009000000ffffffffffffffffffffffffffffffffffffffffffffffff, name=normalize_file, pid=35702, memory used=24.22GB) was running was 349.89GB / 368.00GB (0.950793), which exceeds the memory usage threshold of 0.95. Ray killed this worker (ID: 0c44569cb1b9a03a3962367a7d013284c7229c506b73e8398e218dc0) because it was the most recently scheduled task; to see more information about memory usage on this node, use `ray logs raylet.out -ip 10.0.32.18`. To see the logs of the worker, use `ray logs worker-0c44569cb1b9a03a3962367a7d013284c7229c506b73e8398e218dc0*out -ip 10.0.32.18. Top 10 memory users:
PID	MEM(GB)	COMMAND
32635	49.62	
32636	41.63	ray::normalize_file
31471	29.60	ray::normalize_

In [ ]:
import pandas as pd
res_pd =pd.DataFrame(results)
print(
res_pd.query("message !='success'").count(),
res_pd.query("message =='success'").count()
)

In [ ]:
# 3. Inspect paths
result = results[0]
input_file = result['input_path']
output_file = result['output_path']

print(f"Input:  {input_file}")
print(f"Output: {output_file}")
print(f"Rows:   {result['row_count']:,}")

assert input_file != output_file
assert not output_file.startswith(config.data.raw_data_path)
print("✓ Paths validated")

In [ ]:
# 4. Read output (100 records)
data_access = DataAccessFactory.create('s3', region='us-east-1')
output_data = data_access.read(output_file).limit(100).collect()
print(f"✓ Read {len(output_data)} records")
output_data.head()

In [ ]:
# 5. Compare schemas
normalizer = BMLLNormalizer()
expected_schema = normalizer.get_schema('trades')

missing = set(expected_schema.keys()) - set(output_data.columns)
assert not missing, f"Missing columns: {missing}"

for col, expected_type in expected_schema.items():
    actual_type = output_data.schema[col]
    assert actual_type == expected_type, f"{col}: expected {expected_type}, got {actual_type}"

print(f"✓ Schema validated ({len(output_data.columns)} columns)")
print("\n✓ All tests passed!")